# 🚀 10 Products Object Detection Training with YOLO
This notebook trains a custom YOLO Object Detection model to detect and recognize 10 snack products from live video or images.

### 📋 Instructions:
1. In Google Colab, go to **Runtime → Change runtime type → T4 GPU** (Hardware accelerator).
2. Run the cells step-by-step or click **Runtime → Run all**.

### Step 1: Check GPU and Install Dependencies

In [ ]:
!nvidia-smi
!pip install -q ultralytics

### Step 2: Upload or Extract Dataset
Upload  (8.5 MB) to Colab using the file browser on the left, or run the cell below to upload directly.

In [ ]:
import os
import shutil
from google.colab import files

# Upload dataset_with_backgrounds.zip (9.9 MB)
if not os.path.exists("dataset_with_backgrounds.zip"):
    print("Please upload dataset_with_backgrounds.zip (9.9 MB):")
    uploaded = files.upload()

# Unzip dataset
!rm -rf dataset
!unzip -q dataset_with_backgrounds.zip -d dataset
print("Dataset with background negative samples extracted successfully!")

# Display data.yaml
with open("dataset/data.yaml", "r") as f:
    print("\n--- data.yaml configuration ---")
    print(f.read())


### Step 3: Train Custom YOLO Object Detector
We use  (Nano version) — it is super fast, lightweight, and achieves high FPS on CPU/webcams.

In [ ]:
from ultralytics import YOLO

# Load base pre-trained model
model = YOLO("yolo11n.pt")

# Retrain with Negative Backgrounds + Multi-Scale Distance Robustness
results = model.train(
    data="dataset/data.yaml",
    epochs=80,
    imgsz=640,
    batch=16,
    scale=0.7,        # Scales packets 30% - 170% (30-40cm distance support)
    mosaic=1.0,       # Stitches 4 images into one
    close_mosaic=10,  # Turn off mosaic for final 10 epochs for crisp text
    degrees=10,       # Tilt tolerance
    translate=0.1,    # Position shifts
    fliplr=0.0,       # Keeps product brand text unmirrored
    name="product_detector_v3",
    plots=True
)

print("Training complete! Negative background learning finished!")


### Step 4: Validate Model & Inspect Results

In [ ]:
import glob
from IPython.display import Image, display

# Show latest Confusion Matrix
cm_files = glob.glob("runs/detect/*/confusion_matrix.png")
if cm_files:
    latest_cm = sorted(cm_files)[-1]
    print(f"Displaying Confusion Matrix from {latest_cm}:")
    display(Image(filename=latest_cm))

# Show latest Validation Batch Predictions
val_preds = glob.glob("runs/detect/*/val_batch*_pred.jpg")
if val_preds:
    latest_pred = sorted(val_preds)[-1]
    print(f"Sample Predictions on Validation Data from {latest_pred}:")
    display(Image(filename=latest_pred))


### Step 5: Download Your  Weights File
Run this cell to download  directly to your computer. Place it in your project folder to use with your live webcam script!

In [ ]:
from google.colab import files
import glob

# Find latest best.pt weights
weights_files = glob.glob("runs/detect/*/weights/best.pt")
if weights_files:
    latest_weights = sorted(weights_files)[-1]
    print(f"Downloading {latest_weights} to your computer...")
    files.download(latest_weights)
else:
    print("Weights file not found.")